# Module 3 Lab: Rescue a Messy Dataset
**CS 82A — Data Analysis with Python**
**Ramisha Tasfia**

---
## Step 1: Load and Size Up the File

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('messy_sales.csv')

print('Shape:', df.shape)
print()
print('Column types:')
print(df.dtypes)

In [ ]:
df.head(10)

---
## Step 2: First Observations

Three problems visible before any cleaning:

1. **Mixed date formats** — the `date` column contains both `MM/DD/YYYY` and `YYYY-MM-DD` formats in the same column, which means pandas reads it as plain text (`object`) instead of a real date type.

2. **Zip codes stored as integers** — zip codes like `02134` and `02116` are stored as numbers (`2134`, `2116`), so their leading zeros were stripped. Zip codes are identifiers, not numbers, so this type is wrong.

3. **Missing prices** — several rows in the `price` column have no value (`NaN`). Any revenue calculation on this data as-is would silently ignore those orders or error out.

---
## Step 3: Count Missing Values

In [ ]:
df.isna().sum()

---
## Step 4: Impute Missing Prices

In [ ]:
# Use median, not mean — price distributions are typically right-skewed.
# High-ticket items (monitors at $479, $341) pull the mean upward ($56.07),
# while the median ($37.53) better represents the typical transaction.
fill_value = df['price'].median()
print(f'Median fill value used: ${fill_value}')

df['price'] = df['price'].fillna(fill_value)

print(f'Missing prices after fill: {df["price"].isna().sum()}')

In [ ]:
# Checkpoint: 0 missing values everywhere
df.isna().sum()

---
## Step 5: Remove Duplicate Rows

In [ ]:
print(f'Duplicate rows found: {df.duplicated().sum()}')

df = df.drop_duplicates()

print(f'Shape after removing duplicates: {df.shape}')

---
## Step 6: Repair the Zip Codes

In [ ]:
df['zip'] = df['zip'].astype(str).str.zfill(5)

# Verify — should show 02134 and 02116 with leading zeros restored
print('Sample zip codes after repair:')
print(df['zip'].unique()[:10])

---
## Step 7: Standardize the Dates

In [ ]:
df['date'] = pd.to_datetime(df['date'], format='mixed')

print('Column types after date conversion:')
print(df.dtypes)

In [ ]:
# Checkpoint: shape (292, 6), zip is object, date is datetime64
print('Shape:', df.shape)
print('zip dtype:', df['zip'].dtype)
print('date dtype:', df['date'].dtype)

---
## Step 8: Investigate Negative Quantities

In [ ]:
# Display rows with negative qty
df[df['qty'] < 0]

In [ ]:
# Decision: REMOVE these rows.
#
# A completed sales transaction cannot involve a negative number of units sold.
# This dataset has no 'return' flag, no 'transaction_type' column, and no
# separate returns ledger — so there is no way to confirm these are legitimate
# return records rather than data entry errors (e.g., typing -5 instead of 5).
#
# Keeping them would understate units sold and distort any per-product
# volume analysis. Removing them is the conservative, defensible choice.

df = df[df['qty'] >= 0]

print(f'Shape after removing negative qty rows: {df.shape}')

---
## Step 9: Save the Cleaned File

In [ ]:
df.to_csv('sales_clean.csv', index=False)
print('sales_clean.csv saved successfully.')
print(f'Final shape: {df.shape}')

---
## Step 10: Cleaning Log

| Step | Change | Count Before | Count After | Reason |
|------|--------|-------------|-------------|--------|
| Load | Read `messy_sales.csv` | — | 300 rows | Starting point |
| Missing prices | Filled with median ($37.53) | 12 NaN | 0 NaN | Median resists skew from high-ticket items; mean ($56.07) would overstate imputed values by ~49% |
| Duplicates | Removed fully repeated rows | 300 rows | 292 rows | Duplicate orders would double-count revenue and units |
| Zip codes | Converted `int64` to `str`, padded to 5 chars with `.zfill(5)` | `2134`, `2116` | `02134`, `02116` | Zip codes are geographic identifiers, not numbers; storing as int strips leading zeros permanently |
| Dates | Converted mixed-format `object` column to `datetime64` via `pd.to_datetime(..., format='mixed')` | `object` | `datetime64[ns]` | Real datetime type enables date math, filtering by range, and time-series analysis |
| Negative qty | Removed 3 rows with `qty < 0` (order IDs: 1140, 1233, 1025) | 292 rows | 289 rows | No return indicator exists in this dataset; negative unit counts are physically invalid for a sales record and cannot be verified as intentional |

**Final dataset: 289 rows x 6 columns. All five defect classes resolved.**

The use of the mean as opposed to a median to determine an order's missing price will artificially increase that order's imputed price by increasing it from $37.53 to $56.07. This could lead to increased reported sales for these 12 orders, creating the potential to overstate the profitability of products in the middle range of profit categories.

---
## Appendix: AI Use

Used Claude (Anthropic) to assist in drafting this task for my data science course. Steps 2–8 were reviewed and managed by me. Step 10 was written in my own words. Key prompt: "assist in drafting this task for my data science course."